# Duplicate Network Figures

In [ ]:
import json
from pathlib import Path

# Load the data
filtered_materials = Path.cwd() / "data" / "filtered_materials.json"

with filtered_materials.open() as f:
    data = json.load(f)
    # Only take Ir-O
    network_data = {"Ir-O": data["Ir-O"]}
    # The the database list
    database_list = list(network_data["Ir-O"].keys())

print("Loaded Ir-O chemical system. Database list:")
print(database_list)

In [ ]:
# Get the unique materials via duplicate removal
from matcollect.core.duplicate_removal.duplicate_remover import DuplicateRemover

duplicate_remover = DuplicateRemover(network_data, database_list)

duplicate_remover.structures_by_chemsys = duplicate_remover._generate_pymatgen_structures_by_chemsys()
duplicate_remover.truth_matrices = duplicate_remover._get_structure_similarities(duplicate_remover.structures_by_chemsys,
                                                                                 duplicate_remover.tolerances)
unique_structures_by_chemsys = duplicate_remover._remove_duplicate_structures(duplicate_remover.structures_by_chemsys,
                                                                              duplicate_remover.truth_matrices)
duplicate_remover.unique_materials = duplicate_remover._parse_structures(unique_structures_by_chemsys)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import networkx as nx
import numpy as np

PALETTE = [
    "#636EFA", "#EF553B", "#00CC96", "#AB63FA",
    "#FFA15A", "#19D3F3", "#FF6692", "#B6E880"
]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Charter", "Georgia", "DejaVu Serif"],
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
})

def generate_network_figure_publication(
    structures_by_chemsys,
    truth_matrices,
    unique_materials,
    chemsys: str,
    zoom_bbox: tuple[float, float, float, float] = (0.3, 0.3, 0.35, 0.35),
    output_path: str = "network_figure.pdf",
):
    """
    Publication-quality network figure with inset zoom panel.

    Parameters
    ----------
    zoom_bbox : (x0, y0, w, h)
        Bounding box in *data* coordinates of the region to zoom into.
        Tune this manually after a first pass.
    """
    truth_matrix = truth_matrices[chemsys]
    structures   = structures_by_chemsys[chemsys]

    z  = np.array(truth_matrix, dtype=float)
    n  = z.shape[0]
    labels    = [s["database"] + "/" + s["material_id"] for s in structures]
    databases = [s["database"] for s in structures]
    unique_dbs = sorted(set(databases))
    color_map  = {db: PALETTE[i % len(PALETTE)] for i, db in enumerate(unique_dbs)}
    symbols    = [
        "o" if s["material_id"] in unique_materials[chemsys][s["database"]] else "X"
        for s in structures
    ]

    # ── Graph + layout ──────────────────────────────────────────────────────────
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            if z[i, j] == 1:
                G.add_edge(i, j)

    pos = nx.spring_layout(G, seed=42, k=2 / np.sqrt(n) if n > 1 else 1)
    xs  = np.array([pos[i][0] for i in range(n)])
    ys  = np.array([pos[i][1] for i in range(n)])

    # ── Figure: two panels side-by-side ─────────────────────────────────────────
    fig, (ax_main, ax_zoom) = plt.subplots(
        1, 2,
        figsize=(12, 5),       # inches — tweak to journal column width
        gridspec_kw={"width_ratios": [1.6, 1], "wspace": 0.03},
    )

    def _draw_network(ax, node_size=60, font_size=8, with_labels=False, label_weight='bold'):
        figure_labels = [s["material_id"] for s in structures]
        # Edges
        for u, v in G.edges():
            ax.plot(
                [pos[u][0], pos[v][0]],
                [pos[u][1], pos[v][1]],
                color="#888888", linewidth=0.8, zorder=1,
            )
        # Nodes per database (for legend handles)
        for db in unique_dbs:
            idx = [i for i, d in enumerate(databases) if d == db]
            for i in idx:
                ax.scatter(
                    pos[i][0], pos[i][1],
                    s=node_size,
                    color=color_map[db],
                    marker=symbols[i],
                    edgecolors="white",
                    linewidths=0.4,
                    zorder=2,
                )
        if with_labels:
            for i in range(n):
                ax.annotate(
                    figure_labels[i],
                    xy=(pos[i][0], pos[i][1]),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=font_size,
                    fontfamily="serif",
                    weight=label_weight,
                    color="black",
                )

    # Main network draw call
    _draw_network(ax_main, node_size=50, font_size=9, with_labels=False)
    ax_main.set_title(f"Duplicate network — {chemsys}", fontsize=14, pad=6, fontfamily="serif")

    ax_main.set_aspect("equal")
    ax_main.axis("off")

    # ── Zoom bounding box drawn on main panel ────────────────────────────────────
    x0z, y0z, wz, hz = zoom_bbox
    rect = mpatches.FancyBboxPatch(
        (x0z, y0z), wz, hz,
        boxstyle="square,pad=0",
        linewidth=1.6,
        edgecolor="#E8593C",
        facecolor="none",
        linestyle="--",
        transform=ax_main.transData,
        zorder=5,
    )
    ax_main.add_patch(rect)

    # ── Zoom panel ───────────────────────────────────────────────────────────────
    _draw_network(ax_zoom, node_size=100, font_size=11, with_labels=True)
    ax_zoom.set_xlim(x0z, x0z + wz)
    ax_zoom.set_ylim(y0z, y0z + hz)
    ax_zoom.set_title("Close-up of duplicate cluster", fontsize=14, pad=6, fontfamily="serif")
    ax_zoom.set_aspect("equal")
    ax_zoom.spines[:].set_linewidth(1.6)
    for spine in ax_zoom.spines.values():
        spine.set_edgecolor("#E8593C")
    ax_zoom.set_xticks([])
    ax_zoom.set_yticks([])

    # ── Adjust layout to make room for a figure‑level legend below the panels ───
    fig.tight_layout(pad=0.8)          # initial automatic layout
    fig.subplots_adjust(bottom=0.2)    # increase bottom margin → subplots move up
    fig.canvas.draw()                  # finalise all transforms

    # ── Connection lines between panels (figure‑level coords) ───────────────────
    def _data_to_fig(ax, xy):
        return fig.transFigure.inverted().transform(
            ax.transData.transform(xy)
        )

    for corner_main, corner_zoom in [
        ((x0z + wz, y0z + hz), (0.0, 1.0)),  # top-right of bbox → top-left of zoom
        ((x0z + wz, y0z),      (0.0, 0.0)),  # bottom-right → bottom-left
    ]:
        p1 = _data_to_fig(ax_main, corner_main)
        p2_ax = ax_zoom.get_position()
        p2 = np.array([
            p2_ax.x0 + corner_zoom[0] * p2_ax.width,
            p2_ax.y0 + corner_zoom[1] * p2_ax.height,
        ])
        line = matplotlib.lines.Line2D(
            [p1[0], p2[0]], [p1[1], p2[1]],
            transform=fig.transFigure,
            color="#E8593C", linewidth=1.0, linestyle="--",
            zorder=10,
        )
        fig.add_artist(line)

    # ── Figure‑level legend (placed below the zoom panel) ────────────────────────
    DB_DISPLAY_NAMES = {
                "https://alexandria.icams.rub.de/pbe": "Alexandria",
                "https://optimade.materialsproject.org/": "The Materials Project",
                "https://oqmd.org/optimade/": "OQMD",
                }
    db_handles = [
        mlines.Line2D(
            [],
            [],
            color=color_map[db],
            marker="s",
            linestyle="none",
            markersize=5,
            label=DB_DISPLAY_NAMES.get(db, db)
        )
        for db in unique_dbs
    ]
    marker_handles = [
        mlines.Line2D([], [], color="gray", marker="o", linestyle="none",
                      markersize=5, label="selected (highest-priority source)"),
        mlines.Line2D([], [], color="gray", marker="X", linestyle="none",
                      markersize=5, label="duplicate (lower-priority)"),
        mlines.Line2D([], [], color="#888888", linewidth=0.8,
                      label="duplicate pair"),
    ]
    # Ensure the order column-wise:
    ordered_handles = []
    for db_handle, marker_handle in zip(db_handles, marker_handles, strict=False):
        ordered_handles.append(db_handle)
        ordered_handles.append(marker_handle)

    # Legend
    fig.legend(
        handles=ordered_handles,
        loc='lower center',
        bbox_to_anchor=(0.556, 0.02),
        ncol=3,
        prop={"family": "serif", "size": 11},
        framealpha=0.85,
        edgecolor="0.7",
    )

    # Panel labels
    for ax, label in [(ax_main, "(a)"), (ax_zoom, "(b)")]:
        ax.text(
            0.01, 0.99, label,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=13, fontweight="bold",
            fontfamily="serif",
        )


    fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)
    print(f"Saved → {output_path}")
    return fig

In [ ]:
# First pass — plot to screen to find the right bbox
fig = generate_network_figure_publication(
    duplicate_remover.structures_by_chemsys,
    duplicate_remover.truth_matrices,
    duplicate_remover.unique_materials,
    chemsys="Ir-O",         # pick one chemsys
    zoom_bbox=(0.558, -0.16, 0.082, 0.08),  # x0, y0, width, height in data coords
    output_path="figures/Ir-O_network.pdf",
)
plt.show()  # inspect and tune zoom_bbox

# Pourbaix Figures

In [ ]:
import json
from pathlib import Path

# Load the data
filtered_materials = Path.cwd() / "data" / "unique_materials.json"

with filtered_materials.open() as f:
    data = json.load(f)
    # Only take Ir-O
    stability_data = {"Ir-O": data["Ir-O"]}

print("Loaded Ir-O chemical system. Database list:")

In [ ]:
# Conduct the stability analysis
import os

from dotenv import load_dotenv

from matcollect.core.stability_analysis.pourbaix_analyzer import PourbaixAnalyzer

load_dotenv()  # loads variables from .env
mp_api_key = os.getenv("MP_API_KEY")
U, PH = 1.8, 1.0

pourbaix_analyzer = PourbaixAnalyzer(stability_data, mp_api_key)

pourbaix_analyzer.pourbaix_materials = pourbaix_analyzer._get_pourbaix_materials(pourbaix_analyzer.materials_dict)
print("Pourbaix materials chemsys:", list(pourbaix_analyzer.pourbaix_materials.keys()))

composition_groups = pourbaix_analyzer._get_pourbaix_composition_groups(pourbaix_analyzer.pourbaix_materials)
print("Composition group identifiers:", list(composition_groups["Ir-O"].keys()))

pourbaix_analyzer.pourbaix_diagrams = pourbaix_analyzer._construct_pourbaix_diagrams(pourbaix_analyzer.mp_api_key, composition_groups)
print("Pourbaix diagram identifiers:", list(pourbaix_analyzer.pourbaix_diagrams.get("Ir-O", {}).keys()))

updated_composition_groups = pourbaix_analyzer._get_decomposition_energies(composition_groups, PH, U)
print("Sample decomp energies:", [(e["material_id"], e["decomposition_energy_per_atom"])
      for e in list(updated_composition_groups["Ir-O"].values())[0]["entries"]][:3])

updated_materials_dict = pourbaix_analyzer._parse_structures(updated_composition_groups)
print("Updated materials count:", sum(len(db) for db in updated_materials_dict["Ir-O"].values()))

In [ ]:
import math
from collections import defaultdict

import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt
from pymatgen.analysis.pourbaix_diagram import PourbaixEntry, PourbaixPlotter


def get_pourbaix_plotters(pourbaix_diagrams, composition_groups: dict, ph: float, u: float) -> dict:
    pourbaix_plots = defaultdict(lambda: defaultdict(dict))
    for chemsys, chemsys_diagrams in pourbaix_diagrams.items():
        for identifier, composition_diagram in chemsys_diagrams.items():
            plotter = PourbaixPlotter(composition_diagram)
            for entry in composition_groups[chemsys][identifier]["entries"]:
                if entry["material_id"] != "mp-2723":
                    continue
                database = entry["database"]
                pbx_entry = entry["pourbaix_entry"]
                decomp_energy = entry["decomposition_energy_per_atom"]
                if pbx_entry is None:
                    continue
                fig, _ = create_entry_stability_figure(
                    entry["material_id"], pbx_entry, plotter, decomp_energy, ph, u
                    )
                pourbaix_plots[chemsys][database][entry["material_id"]] = fig
                plt.close(fig)
    return pourbaix_plots

def create_entry_stability_figure(material_id: str,
                                  entry: PourbaixEntry,
                                  plotter: PourbaixPlotter,
                                  decomposition_energy_per_atom: float,
                                  ph: float,
                                  u: float) -> tuple[plt.Figure, plt.Axes]:
    mpl.rcParams['font.family'] = 'serif'
    mpl.rcParams['font.serif'] = ['Charter', 'Georgia', 'DejaVu Serif']

    # Sanity check: clip ph and U
    ph = max(-2, min(16, ph))
    u = max(-3, min(3, u))
    # Compute the bounds for y-axis
    u_min_axis = min(-3, math.floor(u))
    u_max_axis = max(3, math.ceil(u))
    # Compute the bounds for the x-axis
    ph_min_axis = min(-2, math.floor(ph))
    ph_max_axis = max(16, math.ceil(ph))

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    plotter.plot_entry_stability(entry,
                                ax=ax,
                                pH_range=(ph_min_axis, ph_max_axis),
                                V_range=(u_min_axis, u_max_axis),
                                pH_resolution=100,
                                V_resolution=100,
                                show_neutral_axes=False,
                                cmap="YlOrRd",
                                e_hull_max=1.0,
                                )

    # Fix hairlines in PDF export
    for collection in ax.collections:
        collection.set_linewidth(0)
        collection.set_edgecolor("none")
        collection.set_rasterized(True)

    # Dynamic axis limits
    ax.set_xlim([ph_min_axis, ph_max_axis])
    ax.set_ylim([u_min_axis, u_max_axis])
    # Ticks
    ph_ticks = range(ph_min_axis, ph_max_axis + 1, 2)
    u_ticks = range(u_min_axis, u_max_axis + 1, 1)
    ax.set_yticks(u_ticks)
    ax.set_xticks(ph_ticks)
    # Labels
    ax.set_xlabel("pH", fontsize=26)
    ax.set_ylabel("Applied Potential (V vs. SHE)", fontsize=26)
    ax.tick_params(axis="both", which="major", labelsize=24)
    for txt in ax.texts:
        txt.set_visible(False)
    for i, line in enumerate(ax.get_lines()):
        if i == 0:
            line.set_color("#AE86E2")
            line.set_alpha(0.8)
            line.set_linestyle("-")
            line.set_label("Hydrogen Stability Line")
            line.set_linewidth(4)
        elif i == 1:
            line.set_color("#86E2DD")
            line.set_alpha(0.8)
            line.set_linestyle("-")
            line.set_label("Oxygen Stability Line")
            line.set_linewidth(4)
        else:
            line.set_visible(False)

    # Dot marking the (pH, U) operating condition
    ax.plot(ph, u,
            marker="o",
            markersize=8,
            color="white",
            markeredgecolor="black",
            markeredgewidth=1.5,
            zorder=5,
            label=f"Operating point (pH={ph:.1f}, U={u:.1f} V)")

    # --- Add decomposition energy annotation ---
    # Choose a suitable offset
    offset_x = 0.5   # pH units
    offset_y = 0.0   # V units
    ax.annotate(
        f"$\\Delta G_\\mathrm{{decomp}}$ = {decomposition_energy_per_atom:.2f} eV atom$^{{-1}}$",
        xy=(ph, u),
        xytext=(ph + offset_x, u + offset_y),
        fontsize=22,
        fontfamily="serif",
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.7},
    )

    ax.legend(fontsize=22, loc="lower right")
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=24)
    cbar.ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    cbar.set_label(f"$\\Delta G_\\mathrm{{decomp}}$ for {material_id} (eV atom$^{{-1}}$)",fontsize=26)
    fig.tight_layout()
    return fig, ax

# Save
plotter = get_pourbaix_plotters(pourbaix_analyzer.pourbaix_diagrams, updated_composition_groups, PH, U)


plotter["Ir-O"]["https://optimade.materialsproject.org/"]["mp-2723"].savefig("figures/mp-2723_pourbaix_stability.pdf",
                                                                                   format="pdf",
                                                                                   bbox_inches="tight")


# Energy Above Hull

In [ ]:
import json
from pathlib import Path

# Load the data
filtered_materials = Path.cwd() / "data" / "unique_materials.json"

with filtered_materials.open() as f:
    data = json.load(f)
    # Only take Ir-O and Ir-O-Ru
    stability_data = {"Ir-O": data["Ir-O"], "Ir-O-Ru": data["Ir-O-Ru"]}

# Conduct the stability analysis
from itertools import combinations

from plotly.subplots import make_subplots
from pymatgen.analysis.phase_diagram import PDPlotter, PhaseDiagram

from matcollect.core.stability_analysis.e_above_hull_analyzer import HullAnalyzer

hull_analyzer = HullAnalyzer(stability_data)

# ── Publication style parameters ────────────────────────────────────────────
FONT_FAMILY     = "Georgia, DejaVu Serif, serif"
FONT_SIZE_TITLE = 34      # axis titles and colorbar title
FONT_SIZE_TICK  = 34      # axis ticks and colorbar ticks
FONT_SIZE_LABEL = 34      # point labels (IrO2, Ir, O, etc.)
FONT_SIZE_LEGEND= 30      # legend text
LINE_WIDTH      = 8       # convex hull line
MARKER_SIZE     = 22      # scatter point size
COLORBAR_WIDTH  = 25      # colorbar thickness in pixels
FIG_WIDTH       = 1100    # figure width in pixels
FIG_HEIGHT      = 800     # figure height in pixels
# ────────────────────────────────────────────────────────────────────────────

def analyze(hull_analyzer: HullAnalyzer, energy_visibility_threshold: float = 0.5) -> tuple[dict, dict]:
    pd_entries = {}
    for chemsys, chemsys_materials in hull_analyzer.materials_dict.items():
        if len(chemsys.split("-")) > 3:  # noqa: PLR2004
            continue
        entries = hull_analyzer._build_pd_entries(chemsys_materials)  # noqa: SLF001
        list_of_elements = chemsys.split("-")
        all_chemsys = []
        for r in range(1, len(list_of_elements) + 1):
            for combo in combinations(list_of_elements, r):
                all_chemsys.append(sorted(combo))  # noqa: PERF401
        terminal_entries = []
        for possible_chemsys in all_chemsys:
            if possible_chemsys == list_of_elements:
                continue
            possible_chemsys_str = "-".join(possible_chemsys)
            terminal_entries.extend(hull_analyzer.terminal_entries[possible_chemsys_str])
        phase_diagram = PhaseDiagram(entries + terminal_entries)
        plotter = PDPlotter(phase_diagram, show_unstable=energy_visibility_threshold)
        fig = plotter.get_plot()
        apply_figure_layout(fig)
        hull_analyzer._update_hull_energies(phase_diagram, entries)  # noqa: SLF001
        pd_entries[chemsys] = entries
        hull_analyzer.hull_phase_diagrams[chemsys] = fig
    return hull_analyzer.hull_phase_diagrams


def apply_figure_layout(fig) -> None:
    """Apply consistent layout styling to a phase diagram figure."""
    common_axis = {
        "mirror": False,
        "showgrid": False,
        "ticks": "outside",
        "tickfont": {"size": FONT_SIZE_TICK, "color": "black"},
        "title_font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
    }
    layout_kwargs = {
        "template": "simple_white",
        "autosize": False,
        "height": FIG_HEIGHT,
        "width": FIG_WIDTH,
        "paper_bgcolor": "white",
        "font": {
            "size": FONT_SIZE_TITLE,
            "color": "black",
            "family": FONT_FAMILY,
        },
        "legend": {
            "font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
            "title_font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
            "x": 0,
            "y": 1.1,
        },
        "margin": {"l": 20, "r": 20, "t": 40, "b": 40}
    }
    layout_kwargs["xaxis"] = {**common_axis, "dtick": 0.2}
    layout_kwargs["yaxis"] = {**common_axis, "title": "Formation energy (eV atom⁻¹)"}

    # Markers and lines
    fig.update_traces(
        marker={"size": MARKER_SIZE},
        line={"width": LINE_WIDTH},
    )

    # Point labels (IrO2, IrO3, etc.)
    for trace in fig.data:
        if hasattr(trace, "textfont"):
            trace.textfont = {"size": FONT_SIZE_LABEL, "family": FONT_FAMILY}

    # Endpoint annotations (Ir, O)
    for annotation in fig.layout.annotations:
        if annotation.text and annotation.text != "":
            annotation.font = {"size": FONT_SIZE_LABEL, "family": FONT_FAMILY}

    # Colorbar on unstable points trace
    for trace in fig.data:
        if hasattr(trace, "marker") and hasattr(trace.marker, "colorscale") and trace.marker.colorscale:
            trace.marker.showscale = True
            trace.marker.colorbar = {
                "title": {
                    "text": "Energy above hull (eV atom⁻¹)",
                    "font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
                    "side": "right",
                },
                "tickfont": {"size": FONT_SIZE_TICK, "family": FONT_FAMILY},
                "thickness": COLORBAR_WIDTH,
                "len": 0.75,
                "tick0": 0.0,
                "dtick": 0.1,
            }
            break

    # Remove the "Labels" legend item
    for trace in fig.data:
        if hasattr(trace, "name") and trace.name and "label" in trace.name.lower():
            trace.showlegend = False

    fig.update_layout(**layout_kwargs)


hull_analyzer.hull_phase_diagrams = analyze(hull_analyzer,
                                            energy_visibility_threshold=0.5)

fig = hull_analyzer.hull_phase_diagrams["Ir-O"]
output_path = "figures/Ir-O_phase_diagram.pdf"
fig.write_image(output_path, format="pdf")
print(f"Saved → {output_path}")

In [ ]:
import json
from pathlib import Path

# Load the data
filtered_materials = Path.cwd() / "data" / "unique_materials.json"

with filtered_materials.open() as f:
    data = json.load(f)
    # Only take Ir-O and Ir-O-Ru
    stability_data = {"Ir-O-Ru": data["Ir-O-Ru"]}

# Conduct the stability analysis
from itertools import combinations

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pymatgen.analysis.phase_diagram import PDPlotter, PhaseDiagram

from matcollect.core.stability_analysis.e_above_hull_analyzer import HullAnalyzer

# ── Publication style parameters ────────────────────────────────────────────
FONT_FAMILY      = "Georgia, DejaVu Serif, serif"
FONT_SIZE_TITLE  = 34      # axis titles and colorbar title
FONT_SIZE_TICK   = 34      # axis ticks and colorbar ticks
FONT_SIZE_LABEL  = 34      # point labels (IrO2, Ir, O, etc.)
FONT_SIZE_LEGEND = 30      # legend text
LINE_WIDTH       = 8       # convex hull line
MARKER_SIZE      = 22      # scatter point size
COLORBAR_WIDTH   = 25      # colorbar thickness in pixels
COLORBAR_DTICK   = 0.1     # colorbar tick interval
FIG_WIDTH        = 1100    # figure width in pixels
FIG_HEIGHT       = 800     # figure height in pixels
# ────────────────────────────────────────────────────────────────────────────

def analyze(hull_analyzer: HullAnalyzer, energy_visibility_threshold: float = 0.5) -> dict:
    pd_entries = {}
    for chemsys, chemsys_materials in hull_analyzer.materials_dict.items():
        if len(chemsys.split("-")) > 3:  # noqa: PLR2004
            continue
        entries = hull_analyzer._build_pd_entries(chemsys_materials)  # noqa: SLF001
        list_of_elements = chemsys.split("-")
        all_chemsys = []
        for r in range(1, len(list_of_elements) + 1):
            for combo in combinations(list_of_elements, r):
                all_chemsys.append(sorted(combo))  # noqa: PERF401
        terminal_entries = []
        for possible_chemsys in all_chemsys:
            if possible_chemsys == list_of_elements:
                continue
            possible_chemsys_str = "-".join(possible_chemsys)
            terminal_entries.extend(hull_analyzer.terminal_entries[possible_chemsys_str])
        phase_diagram = PhaseDiagram(entries + terminal_entries)
        plotter = PDPlotter(phase_diagram, show_unstable=energy_visibility_threshold)
        fig = plotter.get_plot()
        apply_figure_layout(fig, list_of_elements)
        hull_analyzer._update_hull_energies(phase_diagram, entries)  # noqa: SLF001
        pd_entries[chemsys] = entries
        hull_analyzer.hull_phase_diagrams[chemsys] = fig
    return hull_analyzer.hull_phase_diagrams


def apply_figure_layout(fig, list_of_elements: list) -> None:
    """Apply consistent layout styling to a phase diagram figure."""
    common_axis = {
        "mirror": False,
        "showgrid": False,
        "ticks": "outside",
        "tickfont": {"size": FONT_SIZE_TICK, "color": "black"},
        "title_font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
    }
    ternary_axis = {
        "showgrid": True,
        "ticks": "outside",
        "tickfont": {"size": FONT_SIZE_TICK, "color": "black"},
        "title_font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
    }
    layout_kwargs = {
        "template": "simple_white",
        "autosize": False,
        "height": FIG_HEIGHT,
        "width": FIG_WIDTH,
        "paper_bgcolor": "white",
        "font": {
            "size": FONT_SIZE_TITLE,
            "color": "black",
            "family": FONT_FAMILY,
        },
        "legend": {
            "font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
            "title_font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
            "x": 0,
            "y": 1.1,
        },
        "margin": {"l": 20, "r": 20, "t": 40, "b": 40}
    }

    # Markers and lines
    fig.update_traces(
        marker={"size": MARKER_SIZE},
        line={"width": LINE_WIDTH},
    )

    # Point labels
    for trace in fig.data:
        if hasattr(trace, "textfont"):
            trace.textfont = {"size": FONT_SIZE_LABEL, "family": FONT_FAMILY}

    # Endpoint annotations
    for annotation in fig.layout.annotations:
        if annotation.text and annotation.text != "":
            annotation.font = {"size": FONT_SIZE_LABEL, "family": FONT_FAMILY}

    if len(list_of_elements) == 3:  # noqa: PLR2004
        apply_ternary_layout(fig, layout_kwargs, ternary_axis)
    else:
        layout_kwargs["xaxis"] = {**common_axis, "dtick": 0.2}
        layout_kwargs["yaxis"] = {**common_axis, "title": "Formation energy (eV atom⁻¹)"}
        # Colorbar on unstable points trace (binary only)
        for trace in fig.data:
            if hasattr(trace, "marker") and hasattr(trace.marker, "colorscale") and trace.marker.colorscale:
                trace.marker.showscale = True
                trace.marker.colorbar = {
                    "title": {
                        "text": "Energy above hull (eV atom⁻¹)",
                        "font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
                        "side": "right",
                    },
                    "tickfont": {"size": FONT_SIZE_TICK, "family": FONT_FAMILY},
                    "thickness": COLORBAR_WIDTH,
                    "len": 0.75,
                    "tick0": 0.0,
                    "dtick": COLORBAR_DTICK,
                }
                break

    fig.update_layout(**layout_kwargs)


def apply_ternary_layout(fig, layout_kwargs: dict, ternary_axis: dict) -> None:
    """Apply ternary-specific layout overrides."""
    layout_kwargs["width"] = FIG_WIDTH
    layout_kwargs["ternary"] = {
        "aaxis": {**ternary_axis, "dtick": 0.2},
        "baxis": {**ternary_axis, "dtick": 0.2},
        "caxis": {**ternary_axis, "dtick": 0.2},
    }
    layout_kwargs["legend"] = {
        "font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
        "title_font": {"size": FONT_SIZE_LEGEND, "family": FONT_FAMILY},
        "x": 0.05,
        "y": 1.0,
        "xanchor": "left",
        "yanchor": "top",
    }
    layout_kwargs["margin"] = {
        "t": 80, "b": 80, "l": 0, "r": 0, "autoexpand": True
    }
    fig.update_traces(
        marker={"size": MARKER_SIZE},
        line={"width": LINE_WIDTH},
        selector={"type": "scatterternary"}
    )
    for trace in fig.data:
        if hasattr(trace, "marker") and hasattr(trace.marker, "colorscale") and trace.marker.colorscale:
            trace.marker.showscale = True
            trace.marker.colorbar = {
                "x": 0.9,
                "xanchor": "left",
                "y": 0.5,
                "yanchor": "middle",
                "title": {
                    "text": "Energy above hull (eV atom⁻¹)",
                    "font": {"size": FONT_SIZE_TITLE, "family": FONT_FAMILY},
                    "side": "right",
                },
                "tickfont": {"size": FONT_SIZE_TICK, "family": FONT_FAMILY},
                "tick0": 0.0,
                "dtick": COLORBAR_DTICK,
                "thickness": COLORBAR_WIDTH,
                "len": 0.75,
            }
            break

hull_analyzer.hull_phase_diagrams = analyze(hull_analyzer, energy_visibility_threshold=0.5)

fig = hull_analyzer.hull_phase_diagrams["Ir-O-Ru"]
output_path = "figures/Ir-O-Ru_phase_diagram.pdf"
fig.write_image(output_path, format="pdf")
print(f"Saved → {output_path}")

# Cross-Database Calibration Figure

In [ ]:
import json
from pathlib import Path

# Load the data
extracted_materials = Path.cwd() / "data" / "extracted_materials.json"

with extracted_materials.open() as f:
    stability_data = json.load(f)

reference_database = "https://optimade.materialsproject.org/"
database_priority_list = ['https://optimade.materialsproject.org/', 'https://alexandria.icams.rub.de/pbe', 'https://oqmd.org/optimade/']

In [ ]:
# Calibrate Energy First
from matcollect.core.stability_analysis.energy_calibrator import EnergyCalibrator

calibrator = EnergyCalibrator(stability_data, reference_database)
energy_corrected_data = calibrator.calibrate()

In [ ]:
import numpy as np
import networkx as nx

def tag_duplicate_structures(structures_by_chemsys: dict["str", list],
                             truth_matrices: dict["str", np.ndarray]) -> dict[str, list]:
    tagged_structures_by_chemsys = {}
    for chemsys, list_of_structures in structures_by_chemsys.items():
        truth_matrix = truth_matrices[chemsys]
        n = len(list_of_structures)
        # Build graph and find connected components
        G = nx.Graph()  # noqa: N806
        G.add_nodes_from(range(n))
        for i in range(n):
            for j in range(i + 1, n):
                if truth_matrix[i, j]:
                    G.add_edge(i, j)
        # Tag structures based on connected components
        for component in nx.connected_components(G):
            original_idx = min(component)
            original = list_of_structures[original_idx]
            for idx in component:
                structure = list_of_structures[idx]
                if len(component) == 1:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = None
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                elif idx == original_idx:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "original"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                else:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "duplicate"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = original["json_entry"]["normalized_attributes"]["material_id"]
                    structure["json_entry"]["normalized_attributes"]["original_database"] = original["json_entry"]["normalized_attributes"]["database"]
        tagged_structures_by_chemsys[chemsys] = list_of_structures
    return tagged_structures_by_chemsys

In [ ]:
# Tag duplicates and originals
from matcollect.core.duplicate_removal.duplicate_remover import DuplicateRemover

duplicate_remover = DuplicateRemover(energy_corrected_data, database_priority_list)

structures_by_chemsys = duplicate_remover._generate_pymatgen_structures_by_chemsys()
duplicate_remover.truth_matrices = duplicate_remover._get_structure_similarities(structures_by_chemsys,
                                                       duplicate_remover.tolerances)
tagged_structures_by_chemsys = tag_duplicate_structures(structures_by_chemsys,
                                                        duplicate_remover.truth_matrices)
tagged_materials = duplicate_remover._parse_structures(tagged_structures_by_chemsys)

In [ ]:
import json

# Save the tagged materials
output_path = Path.cwd() / "data" / "tagged_materials.json"

with output_path.open("w") as f:
    json.dump(tagged_materials, f, indent=4)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from pathlib import Path

def plot_duplicate_energies(materials_dict: dict, reference_database: str):
    ref_energies_uncorrected = []
    dup_energies_uncorrected = []
    ref_energies_corrected = []
    dup_energies_corrected = []

    for chemsys, chemsys_dict in materials_dict.items():
        for database, database_dict in chemsys_dict.items():
            for material_id, entry in database_dict.items():
                attributes = entry["normalized_attributes"]
                if attributes["duplicate_status"] != "duplicate":
                    continue
                if attributes["original_database"] != reference_database:
                    continue
                original_attributes = (
                    chemsys_dict
                    [attributes["original_database"]]
                    [attributes["original_material_id"]]
                    ["normalized_attributes"]
                )
                ref_energies_uncorrected.append(original_attributes["formation_energy_per_atom_uncorrected"])
                dup_energies_uncorrected.append(attributes["formation_energy_per_atom_uncorrected"])
                ref_energies_corrected.append(original_attributes["formation_energy_per_atom"])
                dup_energies_corrected.append(attributes["formation_energy_per_atom"])

    diffs_uncorrected = np.abs(np.array(dup_energies_uncorrected) - np.array(ref_energies_uncorrected))
    diffs_corrected = np.abs(np.array(dup_energies_corrected) - np.array(ref_energies_corrected))
    vmin = 0
    vmax = max(diffs_uncorrected.max(), diffs_corrected.max())

    def rmse(a, b):
        return np.sqrt(np.mean((np.array(a) - np.array(b)) ** 2))

    rmse_uncorrected = rmse(ref_energies_uncorrected, dup_energies_uncorrected)
    rmse_corrected = rmse(ref_energies_corrected, dup_energies_corrected)

    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Charter", "Georgia", "DejaVu Serif"],
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.direction": "in",
        "ytick.direction": "in",
    })

    cmap = mpl.colormaps["plasma"]
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.5))

    for ax, ref_energies, dup_energies, diffs, title, rmse_val in [
        (ax1, ref_energies_uncorrected, dup_energies_uncorrected, diffs_uncorrected, "Before calibration", rmse_uncorrected),
        (ax2, ref_energies_corrected,   dup_energies_corrected,   diffs_corrected,   "After calibration",  rmse_corrected),
    ]:
        sc = ax.scatter(ref_energies, dup_energies, c=diffs, cmap=cmap, norm=norm,
                        s=60, alpha=0.9, linewidths=0.8, edgecolors="black",
                        zorder=3)
        # Axis ticks
        ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))

        all_energies = ref_energies + dup_energies
        min_e, max_e = min(all_energies), max(all_energies)
        ax.plot([min_e, max_e], [min_e, max_e], color="#666666", linewidth=1.5, linestyle=(0, (4, 3)))

        ax.text(
            0.05, 0.95,
            f"RMSE = {rmse_val:.4f} eV atom$^{{-1}}$",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=9,
            fontfamily="sans-serif",
            bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "0.8", "lw": 0.6},
        )

        ax.set_xlabel(r"Reference $E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_ylabel(r"Duplicate $E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_title(title, fontfamily="sans-serif")

    plt.tight_layout()

    cbar_ax = fig.add_axes([1.01, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
    cbar.set_label(r"$|\Delta E_\mathrm{f}|$ (eV atom$^{-1}$)", fontfamily="sans-serif", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    return fig

fig = plot_duplicate_energies(tagged_materials, reference_database)
output_path = Path.cwd() / "figures" / "cross_database_comparison.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)

# Cross-Database Calibration Figure - Cross Validation

In [ ]:
"""Cross-database energy calibration via structure-matched compounds.

Fits per-(database, element) offsets so formation energies from different
DFT databases sit on one reference scale. Uses stratified k-fold
cross-validation (Huber regression) to get a robust, split-independent
error estimate, then fits the offsets that are actually applied once on
all matched pairs.
"""

import copy
import logging

import numpy as np
from pymatgen.analysis.structure_matcher import StructureMatcher
from sklearn.linear_model import LinearRegression

from matcollect.core.utils.pymatgen_helper import convert_to_structure

logger = logging.getLogger(__name__)


class EnergyCalibrator:
    """Calibrate formation energies across databases using structure-matched pairs.

    Parameters
    ----------
    materials_dict : dict
        Nested dictionary: {chemsys: {database: {material_id: {normalized_attributes: {...}}}}}
    reference_database : str
        The database to use as the energy reference. All other databases
        are corrected onto this one.
    tolerances : dict, optional
        StructureMatcher tolerances. Defaults to ltol=0.1, stol=0.15, angle_tol=3.
    n_splits : int, optional
        Number of folds for k-fold CV. Default 5.
    random_state : int, optional
        Seed for fold assignment.

    Attributes
    ----------
    element_offsets : dict
        Fitted per-(database, element) offsets (eV/atom), from one final fit
        on *all* matched pairs.
    calibration_report : dict
        CV + final-fit statistics, overall and per database.
    matched_pairs_ : list[dict]
        Every matched pair, each carrying:
          - "database", "ref_ef", "other_ef", "delta", "fractions"
          - "residual"     — in-sample residual from the final (all-data) fit
          - "cv_residual"  — out-of-fold residual (unbiased, from a model
                              that never saw this pair during training)
          - "cv_fold"      — which fold held this pair out
    is_trained : bool
        Whether train() has produced offsets.
    """

    def __init__(self,
                 materials_dict: dict,
                 reference_database: str,
                 tolerances: dict | None = None,
                 n_splits: int = 5,
                 random_state: int | None = None):
        self.materials_dict = materials_dict
        self.reference_database = reference_database
        self.tolerances = tolerances or {"ltol": 0.1, "stol": 0.15, "angle_tol": 3}
        self.n_splits = n_splits
        self.random_state = random_state

        self.element_offsets = {}
        self.calibration_report = {}
        self.matched_pairs_ = []
        self.is_trained = False
        self._matcher = StructureMatcher(**self.tolerances)

    def train(self) -> dict:
        """Find matched pairs, run k-fold CV, and fit final offsets on all pairs.

        Read-only w.r.t. materials_dict. Populates element_offsets,
        calibration_report, and matched_pairs_. Call fit() afterwards to
        apply the offsets.
        """
        all_databases = set()
        for chemsys_dict in self.materials_dict.values():
            all_databases.update(chemsys_dict.keys())

        if self.reference_database not in all_databases:
            logger.warning(
                f"Reference database '{self.reference_database}' not found. "  # noqa: G004
                f"Available: {all_databases}. Skipping training."
            )
            return self.calibration_report

        non_ref_databases = sorted(all_databases - {self.reference_database})
        if not non_ref_databases:
            logger.info("Only one database present. No calibration needed.")
            return self.calibration_report

        all_pairs = []
        failed = {}
        for other_db in non_ref_databases:
            pairs = self._find_matched_pairs(self.reference_database, other_db)
            if not pairs:
                logger.warning(
                    f"No matched compounds between '{self.reference_database}' "  # noqa: G004
                    f"and '{other_db}'. This database will be skipped."
                )
                failed[other_db] = {"status": "failed", "reason": "no matched compounds", "n_matches": 0}
                continue
            for pair in pairs:
                pair["database"] = other_db
            all_pairs.extend(pairs)

        if not all_pairs:
            logger.warning("No matched compounds found for any non-reference database.")
            self.calibration_report = {"status": "failed", "per_database": failed}
            return self.calibration_report

        return self._train_kfold(all_pairs, non_ref_databases, failed)

    def fit(self) -> dict:
        """Apply the offsets learned by train() to materials_dict."""
        if not self.is_trained:
            raise RuntimeError(
                "EnergyCalibrator.fit() called before train(). "
                "Call train() first to learn per-element offsets."
            )

        for chemsys_dict in self.materials_dict.values():
            if self.reference_database not in chemsys_dict:
                continue
            for material in chemsys_dict[self.reference_database].values():
                attrs = material["normalized_attributes"]
                ef = attrs.get("formation_energy_per_atom")
                if ef is None:
                    continue
                attrs["formation_energy_per_atom_uncorrected"] = ef
                attrs["calibration_correction"] = 0.0
                attrs["calibration_reference"] = self.reference_database

        databases_with_offsets = {db for db, _ in self.element_offsets}
        for other_db in databases_with_offsets:
            offsets = {el: off for (db, el), off in self.element_offsets.items() if db == other_db}
            self._apply_corrections(other_db, offsets)

        return self.materials_dict

    # --- k-fold CV ---

    def _train_kfold(self, all_pairs: list[dict], non_ref_databases: list[str],
                      failed: dict) -> dict:
        n_splits = self.n_splits
        fold_indices = self._stratified_kfold_indices(all_pairs, n_splits)

        fold_reports = []
        oof_residual: list[float | None] = [None] * len(all_pairs)
        oof_fold: list[int | None] = [None] * len(all_pairs)

        for k in range(n_splits):
            test_idx = fold_indices[k]
            test_idx_set = set(test_idx)
            train_idx = [i for i in range(len(all_pairs)) if i not in test_idx_set]

            train_pairs_fold = [copy.copy(all_pairs[i]) for i in train_idx]
            test_pairs_fold = [copy.copy(all_pairs[i]) for i in test_idx]

            if not test_pairs_fold:
                fold_reports.append({"fold": k, "status": "skipped", "reason": "empty test fold",
                                      "n_train": len(train_pairs_fold), "n_test": 0})
                continue

            offsets_fold, train_report_fold = self._fit_offsets(train_pairs_fold)
            if offsets_fold is None:
                fold_reports.append({"fold": k, "status": "failed", "reason": train_report_fold["reason"],
                                      "n_train": len(train_pairs_fold), "n_test": len(test_pairs_fold)})
                continue

            test_report_fold = self._evaluate(test_pairs_fold, offsets_fold)

            for local_i, global_i in enumerate(test_idx):
                oof_residual[global_i] = test_pairs_fold[local_i]["residual"]
                oof_fold[global_i] = k

            fold_reports.append({
                "fold": k, "status": "success",
                "n_train": len(train_pairs_fold), "n_test": len(test_pairs_fold),
                "train_rmse_meV": train_report_fold["rmse_meV"],
                "test_rmse_meV": test_report_fold.get("rmse_meV"),
            })

        successful_folds = [r for r in fold_reports if r["status"] == "success"]
        if not successful_folds:
            self.calibration_report = {
                "status": "failed",
                "reason": "all folds failed to fit (too few matched pairs per fold "
                          "relative to the number of (database, element) columns)",
                "folds": fold_reports, "per_database": failed,
            }
            return self.calibration_report

        oof_have = [r for r in oof_residual if r is not None]
        cv_rmse_pooled = float(np.sqrt(np.mean(np.square(oof_have)))) if oof_have else float("nan")
        fold_test_rmses = [r["test_rmse_meV"] for r in successful_folds if r.get("test_rmse_meV") is not None]
        cv_rmse_mean = float(np.mean(fold_test_rmses)) if fold_test_rmses else float("nan")
        cv_rmse_std = float(np.std(fold_test_rmses)) if fold_test_rmses else float("nan")

        # Final production offsets: one more fit, on ALL pairs (no holdout).
        final_offsets, final_report = self._fit_offsets([copy.copy(p) for p in all_pairs])
        if final_offsets is None:
            self.calibration_report = {"status": "failed", **final_report, "folds": fold_reports,
                                        "per_database": failed}
            return self.calibration_report

        for i, pair in enumerate(all_pairs):
            pair["cv_residual"] = oof_residual[i]
            pair["cv_fold"] = oof_fold[i]
            db = pair["database"]
            predicted = sum(frac * final_offsets.get((db, el), 0.0) for el, frac in pair["fractions"].items())
            pair["residual"] = pair["delta"] - predicted

        self.element_offsets = final_offsets
        self.matched_pairs_ = all_pairs

        self.calibration_report = {
            "status": "success",
            "reference": self.reference_database,
            "databases": non_ref_databases,
            "n_matches_total": len(all_pairs),
            "cv": {
                "n_splits": n_splits,
                "pooled_rmse_meV": round(cv_rmse_pooled * 1000, 1),
                "fold_rmse_mean_meV": round(cv_rmse_mean, 1),
                "fold_rmse_std_meV": round(cv_rmse_std, 1),
                "folds": fold_reports,
            },
            "final_fit": final_report,
            "per_database": {**self._per_database_breakdown(all_pairs, final_offsets), **failed},
        }

        logger.info(
            f"Calibration {self.reference_database} ↔ {non_ref_databases} "  # noqa: G004
            f"({n_splits}-fold CV): pooled out-of-fold RMSE = {cv_rmse_pooled*1000:.1f} meV/atom, "
            f"fold RMSE = {cv_rmse_mean:.1f} ± {cv_rmse_std:.1f} meV/atom ({len(all_pairs)} pairs total)"
        )

        self.is_trained = bool(self.element_offsets)
        return self.calibration_report

    def _stratified_kfold_indices(self, pairs: list[dict], n_splits: int) -> list[list[int]]:
        """Partition pair indices into n_splits folds, stratified by database."""
        rng = np.random.default_rng(self.random_state)
        by_db: dict[str, list[int]] = {}
        for i, pair in enumerate(pairs):
            by_db.setdefault(pair["database"], []).append(i)

        folds: list[list[int]] = [[] for _ in range(n_splits)]
        for _db, idxs in sorted(by_db.items()):
            idxs = np.array(idxs)
            shuffled = idxs[rng.permutation(len(idxs))]
            if len(shuffled) < n_splits:
                for j, idx in enumerate(shuffled):
                    folds[j % n_splits].append(int(idx))
            else:
                for j, chunk in enumerate(np.array_split(shuffled, n_splits)):
                    folds[j].extend(int(x) for x in chunk)
        return folds

    def _per_database_breakdown(self, all_pairs: list[dict], offsets: dict) -> dict:
        databases = sorted({p["database"] for p in all_pairs})
        breakdown = {}
        for db in databases:
            db_pairs = [p for p in all_pairs if p["database"] == db]
            elements = sorted(el for (d, el) in offsets if d == db)
            cv_residuals = [p["cv_residual"] for p in db_pairs if p.get("cv_residual") is not None]
            final_residuals = [p["residual"] for p in db_pairs]
            breakdown[db] = {
                "elements": elements,
                "offsets_meV": {el: round(offsets[(db, el)] * 1000, 1) for el in elements},
                "n_matches": len(db_pairs),
                "cv_rmse_meV": self._rmse_meV(cv_residuals),
                "final_fit_rmse_meV": self._rmse_meV(final_residuals),
            }
        return breakdown

    # --- matching + fitting primitives ---

    def _find_matched_pairs(self, ref_db: str, other_db: str) -> list[dict]:  # noqa: C901
        matched_pairs = []
        seen_pairs = set()

        for chemsys_dict in self.materials_dict.values():
            if ref_db not in chemsys_dict or other_db not in chemsys_dict:
                continue
            ref_materials = chemsys_dict[ref_db]
            other_materials = chemsys_dict[other_db]

            for ref_id, ref_mat in ref_materials.items():
                ref_attrs = ref_mat["normalized_attributes"]
                ref_ef = ref_attrs.get("formation_energy_per_atom")
                ref_formula = ref_attrs.get("reduced_formula")
                if ref_ef is None or ref_attrs.get("lattice_vectors") is None:
                    continue
                try:
                    ref_struct = convert_to_structure(ref_attrs)
                except (TypeError, KeyError, ValueError):
                    continue

                for other_id, other_mat in other_materials.items():
                    other_attrs = other_mat["normalized_attributes"]
                    other_ef = other_attrs.get("formation_energy_per_atom")
                    other_formula = other_attrs.get("reduced_formula")
                    if other_ef is None or other_attrs.get("lattice_vectors") is None:
                        continue
                    if ref_formula != other_formula:
                        continue
                    pair_key = (ref_id, other_id)
                    if pair_key in seen_pairs:
                        continue
                    try:
                        other_struct = convert_to_structure(other_attrs)
                    except (TypeError, KeyError, ValueError):
                        continue
                    try:
                        if self._matcher.fit(ref_struct, other_struct):
                            seen_pairs.add(pair_key)
                            comp = ref_attrs["composition"]
                            n_total = sum(comp.values())
                            fractions = {el: amt / n_total for el, amt in comp.items()}
                            matched_pairs.append({
                                "ref_id": ref_id, "other_id": other_id, "formula": ref_formula,
                                "ref_ef": ref_ef, "other_ef": other_ef,
                                "delta": ref_ef - other_ef, "fractions": fractions,
                            })
                    except Exception:  # noqa: S112
                        continue
        return matched_pairs

    def _fit_offsets(self, matched_pairs: list[dict]) -> tuple[dict | None, dict]:
        """Jointly fit per-(database, element) offsets via Huber regression.

        Annotates each pair with "residual" (eV/atom) as a side effect.
        """
        columns = sorted({(p["database"], el) for p in matched_pairs for el in p["fractions"]})
        n_cols = len(columns)
        n_matches = len(matched_pairs)

        if n_matches < n_cols:
            return None, {
                "status": "failed",
                "reason": f"underdetermined: {n_matches} matches < {n_cols} (database, element) columns",
                "n_matches": n_matches, "n_elements": n_cols,
            }

        col_index = {col: j for j, col in enumerate(columns)}
        A = np.zeros((n_matches, n_cols))  # noqa: N806
        b = np.zeros(n_matches)
        for i, pair in enumerate(matched_pairs):
            b[i] = pair["delta"]
            db = pair["database"]
            for el, frac in pair["fractions"].items():
                A[i, col_index[(db, el)]] = frac

        model = LinearRegression(fit_intercept=False)
        model.fit(A, b)

        delta = model.coef_
        residuals = b - A @ delta
        rmse = float(np.sqrt(np.mean(residuals ** 2)))
        max_residual = float(np.max(np.abs(residuals)))
        offsets = {col: float(delta[j]) for j, col in enumerate(columns)}

        for i, pair in enumerate(matched_pairs):
            pair["residual"] = float(residuals[i])

        report = {
            "status": "success", "reference": self.reference_database,
            "n_matches": n_matches, "n_columns": n_cols,
            "rmse_meV": round(rmse * 1000, 1), "max_residual_meV": round(max_residual * 1000, 1),
        }
        if rmse > 0.050:  # noqa: PLR2004
            report["warning"] = f"Train RMSE = {rmse*1000:.0f} meV/atom exceeds 50 meV threshold."
        elif rmse > 0.025:  # noqa: PLR2004
            report["warning"] = f"Train RMSE = {rmse*1000:.0f} meV/atom (above 25 meV DFT uncertainty)."

        return offsets, report

    def _evaluate(self, matched_pairs: list[dict], offsets: dict) -> dict:
        """Evaluate fitted offsets on held-out pairs. Annotates "residual"."""
        if not matched_pairs:
            return {"status": "skipped", "reason": "no test pairs available", "n_matches": 0}

        residuals = []
        for pair in matched_pairs:
            db = pair["database"]
            predicted = sum(frac * offsets.get((db, el), 0.0) for el, frac in pair["fractions"].items())
            resid = pair["delta"] - predicted
            pair["residual"] = float(resid)
            residuals.append(resid)

        residuals = np.array(residuals)
        rmse = float(np.sqrt(np.mean(residuals ** 2)))
        report = {"status": "success", "n_matches": len(matched_pairs),
                  "rmse_meV": round(rmse * 1000, 1),
                  "max_residual_meV": round(float(np.max(np.abs(residuals))) * 1000, 1)}
        return report

    @staticmethod
    def _rmse_meV(residuals: list[float]) -> float | None:  # noqa: N802
        if not residuals:
            return None
        residuals = np.array(residuals)
        return round(float(np.sqrt(np.mean(residuals ** 2))) * 1000, 1)

    def _apply_corrections(self, database: str, offsets: dict) -> None:
        for chemsys_dict in self.materials_dict.values():
            if database not in chemsys_dict:
                continue
            for material in chemsys_dict[database].values():
                attrs = material["normalized_attributes"]
                ef = attrs.get("formation_energy_per_atom")
                comp = attrs.get("composition")
                if ef is None or comp is None:
                    continue
                n_total = sum(comp.values())
                correction = sum((amt / n_total) * offsets.get(el, 0.0) for el, amt in comp.items())
                attrs["formation_energy_per_atom_uncorrected"] = ef
                attrs["formation_energy_per_atom"] = ef + correction
                attrs["calibration_correction"] = correction
                attrs["calibration_reference"] = self.reference_database

    def get_report_summary(self) -> str:
        report = self.calibration_report
        lines = [f"Cross-database calibration (reference: {self.reference_database})", "=" * 60]

        if report.get("status") != "success":
            lines.append(f"\nFAILED: {report.get('reason', 'unknown error')}")
            return "\n".join(lines)

        cv, final = report["cv"], report["final_fit"]
        lines.append(f"\n{cv['n_splits']}-fold CV (all {len(report['databases'])} non-reference databases pooled):")
        lines.append(f"  Pooled out-of-fold RMSE: {cv['pooled_rmse_meV']} meV/atom")
        lines.append(f"  Per-fold RMSE:           {cv['fold_rmse_mean_meV']} \u00b1 {cv['fold_rmse_std_meV']} meV/atom")
        n_failed = sum(1 for f in cv["folds"] if f["status"] != "success")
        if n_failed:
            lines.append(f"  WARNING: {n_failed}/{cv['n_splits']} folds failed to fit or were empty")
        lines.append(f"  Final offsets fit on all {report['n_matches_total']} pairs: RMSE = {final['rmse_meV']} meV/atom")
        if "warning" in final:
            lines.append(f"  WARNING (final fit): {final['warning']}")

        for db, db_report in report["per_database"].items():
            lines.append(f"\n{db}:")
            if db_report.get("status") == "failed":
                lines.append(f"  FAILED: {db_report['reason']}")
                continue
            lines.append(f"  Elements fitted: {', '.join(db_report['elements'])}")
            lines.append(f"  Offsets:         {db_report['offsets_meV']} meV/atom")
            lines.append(f"  Matches:         {db_report['n_matches']} pairs")
            lines.append(f"  CV RMSE:         {db_report['cv_rmse_meV']} meV/atom")
            lines.append(f"  Final-fit RMSE:  {db_report['final_fit_rmse_meV']} meV/atom")

        return "\n".join(lines)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np


def plot_calibration_parity(calibrator):
    """1x3 parity plot: uncorrected vs. final fit (in-sample) vs. CV fit (out-of-fold).

    Panels:
      1. "Before Calibration" — raw energies for every matched pair.
      2. "Final Fit" — all pairs, corrected using the final offsets
         (element_offsets, fit on all data). In-sample.
      3. "CV Fit (out-of-fold)" — all pairs, corrected using each pair's
         own out-of-fold prediction (cv_residual), for an honest read on
         how well the offsets generalize.
    """
    if not calibrator.matched_pairs_:
        raise ValueError("No matched pairs found. Did you call calibrator.train() first?")

    all_pairs = calibrator.matched_pairs_
    cv_pairs = [p for p in all_pairs if p.get("cv_residual") is not None]

    def final_corrected_other_ef(pair):
        db = pair["database"]
        correction = sum(frac * calibrator.element_offsets.get((db, el), 0.0)
                          for el, frac in pair["fractions"].items())
        return pair["other_ef"] + correction

    def cv_corrected_other_ef(pair):
        # ref_ef - cv_residual is guaranteed consistent with cv_residual,
        # since cv_residual = ref_ef - (other_ef + fold_correction).
        return pair["ref_ef"] - pair["cv_residual"]

    def rmse(residuals):
        residuals = np.array(residuals)
        return float(np.sqrt(np.mean(residuals ** 2))) if len(residuals) else float("nan")

    panels = [
        (
            "Before Calibration",
            [p["ref_ef"] for p in all_pairs],
            [p["other_ef"] for p in all_pairs],
            [abs(p["delta"]) for p in all_pairs],
            [p["delta"] for p in all_pairs],
            len(all_pairs),
        ),
        (
            "Calibrated (in-sample)",
            [p["ref_ef"] for p in all_pairs],
            [final_corrected_other_ef(p) for p in all_pairs],
            [abs(p["residual"]) for p in all_pairs],
            [p["residual"] for p in all_pairs],
            len(all_pairs),
        ),
        (
            "Calibrated (out-of-fold)",
            [p["ref_ef"] for p in cv_pairs],
            [cv_corrected_other_ef(p) for p in cv_pairs],
            [abs(p["cv_residual"]) for p in cv_pairs],
            [p["cv_residual"] for p in cv_pairs],
            len(cv_pairs),
        ),
    ]

    all_diffs = [d for _, _, _, diffs, _, _ in panels for d in diffs]
    vmin, vmax = 0, (max(all_diffs) if all_diffs else 1.0)

    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Charter", "Georgia", "DejaVu Serif"],
        "axes.labelsize": 11, "axes.titlesize": 12,
        "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "xtick.direction": "in", "ytick.direction": "in",
    })

    cmap = mpl.colormaps["plasma"]
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.5))

    for ax, (title, ref_energies, dup_energies, abs_diffs, signed_diffs, n) in zip(axes, panels, strict=True):
        if n == 0:
            ax.text(0.5, 0.5, "No pairs", ha="center", va="center",
                     transform=ax.transAxes, fontfamily="sans-serif")
            ax.set_title(title, fontfamily="sans-serif")
            continue

        ax.scatter(ref_energies, dup_energies, c=abs_diffs, cmap=cmap, norm=norm,
                   s=60, alpha=0.9, linewidths=0.8, edgecolors="black", zorder=3)
        ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))

        all_energies = ref_energies + dup_energies
        min_e, max_e = min(all_energies), max(all_energies)
        ax.plot([min_e, max_e], [min_e, max_e], color="#666666", linewidth=1.5, linestyle=(0, (4, 3)))

        rmse_val = rmse(signed_diffs)
        ax.text(0.05, 0.95, f"RMSE = {rmse_val:.4f} eV atom$^{{-1}}$",
                transform=ax.transAxes, ha="left", va="top", fontsize=9, fontfamily="sans-serif",
                bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "0.8", "lw": 0.6})

        ax.set_xlabel(r"Reference $E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_ylabel(r"$E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_title(title, fontfamily="sans-serif")

    plt.tight_layout()
    cbar_ax = fig.add_axes([1.01, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
    cbar.set_label(r"$|\Delta E_\mathrm{f}|$ (eV atom$^{-1}$)", fontfamily="sans-serif", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    return fig

In [ ]:
import json
from pathlib import Path

extracted_materials = Path.cwd() / "data" / "extracted_materials.json"
with extracted_materials.open() as f:
    stability_data = json.load(f)

reference_database = "https://optimade.materialsproject.org/"
database_priority_list = [
    "https://optimade.materialsproject.org/",
    "https://alexandria.icams.rub.de/pbe",
    "https://oqmd.org/optimade/",
]

# Calibrate energy across databases (5-fold CV, Huber regression)
calibrator = EnergyCalibrator(stability_data,
                              reference_database,
                              n_splits=5,
                              random_state=42)
calibrator.train()
print(calibrator.get_report_summary())
energy_corrected_data = calibrator.fit()

# Sanity-check the fit: uncorrected vs. final fit (in-sample) vs. CV fit (out-of-fold)
fig = plot_calibration_parity(calibrator)
output_path = Path.cwd() / "figures" / "cross_database_comparison_cv.pdf"
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)


In [ ]:
import numpy as np
import networkx as nx
def tag_duplicate_structures(structures_by_chemsys: dict["str", list],
                             truth_matrices: dict["str", np.ndarray]) -> dict[str, list]:
    tagged_structures_by_chemsys = {}
    for chemsys, list_of_structures in structures_by_chemsys.items():
        truth_matrix = truth_matrices[chemsys]
        n = len(list_of_structures)
        # Build graph and find connected components
        G = nx.Graph()  # noqa: N806
        G.add_nodes_from(range(n))
        for i in range(n):
            for j in range(i + 1, n):
                if truth_matrix[i, j]:
                    G.add_edge(i, j)
        # Tag structures based on connected components
        for component in nx.connected_components(G):
            original_idx = min(component)
            original = list_of_structures[original_idx]
            for idx in component:
                structure = list_of_structures[idx]
                if len(component) == 1:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = None
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                elif idx == original_idx:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "original"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                else:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "duplicate"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = original["json_entry"]["normalized_attributes"]["material_id"]
                    structure["json_entry"]["normalized_attributes"]["original_database"] = original["json_entry"]["normalized_attributes"]["database"]
        tagged_structures_by_chemsys[chemsys] = list_of_structures
    return tagged_structures_by_chemsys


# Tag duplicates and originals
from matcollect.core.duplicate_removal.duplicate_remover import DuplicateRemover
duplicate_remover = DuplicateRemover(energy_corrected_data, database_priority_list)
structures_by_chemsys = duplicate_remover._generate_pymatgen_structures_by_chemsys()
duplicate_remover.truth_matrices = duplicate_remover._get_structure_similarities(structures_by_chemsys,
                                                       duplicate_remover.tolerances)
tagged_structures_by_chemsys = tag_duplicate_structures(structures_by_chemsys,
                                                        duplicate_remover.truth_matrices)
tagged_materials = duplicate_remover._parse_structures(tagged_structures_by_chemsys)

# Leave One Out Calibration

In [ ]:
"""Cross-database energy calibration via structure-matched compounds.

Fits per-(database, element) offsets so formation energies from different
DFT databases sit on one reference scale. Generalization is evaluated by
leave-one-chemsys-out (LOCO) cross-validation: for each chemical system,
offsets are fit on every *other* chemsys's matched pairs and evaluated on
the held-out one. This is important because two matched pairs from the
same chemsys (e.g. two oxidation states of the same metal) would otherwise
leak between train and test under a random split, making the calibration
look better than it really is on unseen chemistry. The offsets actually
applied downstream are fit once on ALL matched pairs — LOCO is purely a
generalization estimate, not the production model.
"""

import logging

import numpy as np
from pymatgen.analysis.structure_matcher import StructureMatcher
from sklearn.linear_model import HuberRegressor

from matcollect.core.utils.pymatgen_helper import convert_to_structure

logger = logging.getLogger(__name__)


class EnergyCalibrator:
    """Calibrate formation energies across databases using structure-matched pairs.

    Parameters
    ----------
    materials_dict : dict
        Nested dictionary: {chemsys: {database: {material_id: {normalized_attributes: {...}}}}}
    reference_database : str
        The database to use as the energy reference. All other databases
        are corrected onto this one.
    tolerances : dict, optional
        StructureMatcher tolerances. Defaults to ltol=0.1, stol=0.15, angle_tol=3.

    Attributes
    ----------
    element_offsets : dict
        Final per-(database, element) offsets (eV/atom), fit on ALL matched
        pairs — these are what fit() applies.
    calibration_report : dict
        LOCO + final-fit statistics, overall and per database.
    matched_pairs_ : list[dict]
        Every matched pair, each carrying:
          - "database", "chemsys", "ref_ef", "other_ef", "delta", "fractions"
          - "residual"     — in-sample residual from the final (all-data) fit
          - "cv_residual"  — LOCO out-of-fold residual (None if its chemsys's
                              fold couldn't be fit, or its elements weren't
                              covered by the training fold)
          - "cv_fold"      — chemsys that was held out to produce cv_residual
    is_trained : bool
        Whether train() has produced offsets.
    """

    def __init__(self,
                 materials_dict: dict,
                 reference_database: str,
                 tolerances: dict | None = None):
        self.materials_dict = materials_dict
        self.reference_database = reference_database
        self.tolerances = tolerances or {"ltol": 0.1, "stol": 0.15, "angle_tol": 3}

        self.element_offsets = {}
        self.calibration_report = {}
        self.matched_pairs_ = []
        self.is_trained = False
        self._matcher = StructureMatcher(**self.tolerances)

    def train(self) -> dict:
        """Find matched pairs, run LOCO cross-validation, and fit final offsets.

        Read-only w.r.t. materials_dict. Populates element_offsets,
        calibration_report, and matched_pairs_. Call fit() afterwards to
        apply the offsets.
        """
        all_databases = set()
        for chemsys_dict in self.materials_dict.values():
            all_databases.update(chemsys_dict.keys())

        if self.reference_database not in all_databases:
            logger.warning(
                f"Reference database '{self.reference_database}' not found. "  # noqa: G004
                f"Available: {all_databases}. Skipping training."
            )
            return self.calibration_report

        non_ref_databases = sorted(all_databases - {self.reference_database})
        if not non_ref_databases:
            logger.info("Only one database present. No calibration needed.")
            return self.calibration_report

        all_pairs = []
        failed = {}
        for other_db in non_ref_databases:
            pairs = self._find_matched_pairs(self.reference_database, other_db)
            if not pairs:
                logger.warning(
                    f"No matched compounds between '{self.reference_database}' "  # noqa: G004
                    f"and '{other_db}'. This database will be skipped."
                )
                failed[other_db] = {"status": "failed", "reason": "no matched compounds", "n_matches": 0}
                continue
            for pair in pairs:
                pair["database"] = other_db
            all_pairs.extend(pairs)

        if not all_pairs:
            logger.warning("No matched compounds found for any non-reference database.")
            self.calibration_report = {"status": "failed", "per_database": failed}
            return self.calibration_report

        return self._train_loco(all_pairs, non_ref_databases, failed)

    def fit(self) -> dict:
        """Apply the offsets learned by train() to materials_dict."""
        if not self.is_trained:
            raise RuntimeError(
                "EnergyCalibrator.fit() called before train(). "
                "Call train() first to learn per-element offsets."
            )

        for chemsys_dict in self.materials_dict.values():
            if self.reference_database not in chemsys_dict:
                continue
            for material in chemsys_dict[self.reference_database].values():
                attrs = material["normalized_attributes"]
                ef = attrs.get("formation_energy_per_atom")
                if ef is None:
                    continue
                attrs["formation_energy_per_atom_uncorrected"] = ef
                attrs["calibration_correction"] = 0.0
                attrs["calibration_reference"] = self.reference_database

        databases_with_offsets = {db for db, _ in self.element_offsets}
        for other_db in databases_with_offsets:
            offsets = {el: off for (db, el), off in self.element_offsets.items() if db == other_db}
            self._apply_corrections(other_db, offsets)

        return self.materials_dict

    # --- leave-one-chemsys-out CV ---

    def _train_loco(self, all_pairs: list[dict], non_ref_databases: list[str],
                     failed: dict) -> dict:
        # Group once so each fold's train set is a fast dict lookup/concat
        # rather than a fresh O(n) filter of all_pairs per chemsys.
        by_chemsys: dict[str, list[dict]] = {}
        for pair in all_pairs:
            by_chemsys.setdefault(pair["chemsys"], []).append(pair)
        chemsystems = sorted(by_chemsys)

        if len(chemsystems) < 2:
            self.calibration_report = {
                "status": "failed",
                "reason": (
                    f"leave_one_chemsys_out requires matched pairs spanning at least 2 "
                    f"chemical systems; found {len(chemsystems)}."
                ),
                "per_database": failed,
            }
            return self.calibration_report

        cv_residual: dict[int, float] = {}   # id(pair) -> residual
        cv_fold: dict[int, str] = {}         # id(pair) -> held-out chemsys
        fold_reports = {}

        for chemsys in chemsystems:
            fold_test = by_chemsys[chemsys]
            fold_train = [p for cs, pairs in by_chemsys.items() if cs != chemsys for p in pairs]

            fold_offsets, _residuals, fold_train_report = self._fit_offsets(fold_train)
            if fold_offsets is None:
                fold_reports[chemsys] = {
                    "status": "skipped",
                    "reason": f"training fold underdetermined: {fold_train_report['reason']}",
                    "n_test": len(fold_test),
                }
                continue

            trained_columns = set(fold_offsets)
            evaluable, n_skipped = [], 0
            for p in fold_test:
                db = p["database"]
                if any((db, el) not in trained_columns for el in p["fractions"]):
                    n_skipped += 1
                else:
                    evaluable.append(p)

            fold_test_report, fold_residuals = self._evaluate(evaluable, fold_offsets)
            for p, r in zip(evaluable, fold_residuals, strict=True):
                cv_residual[id(p)] = r
                cv_fold[id(p)] = chemsys

            fold_reports[chemsys] = {
                "status": fold_test_report.get("status", "success"),
                "n_train": len(fold_train),
                "n_test": len(fold_test),
                "n_test_evaluated": len(evaluable),
                "n_test_skipped": n_skipped,
                "rmse_meV": fold_test_report.get("rmse_meV"),
            }

        n_folds_skipped = sum(1 for r in fold_reports.values() if r["status"] == "skipped")
        n_test_skipped_total = sum(r.get("n_test_skipped", 0) for r in fold_reports.values())

        loco_rmse_meV = self._rmse_meV(list(cv_residual.values()))
        if not cv_residual:
            self.calibration_report = {
                "status": "failed",
                "reason": "no chemsys fold produced any evaluable out-of-fold pairs",
                "folds": fold_reports,
                "per_database": failed,
            }
            return self.calibration_report

        # Final production offsets: one more fit, on ALL pairs (no holdout).
        final_offsets, final_residuals, final_report = self._fit_offsets(all_pairs)
        if final_offsets is None:
            self.calibration_report = {"status": "failed", **final_report, "folds": fold_reports,
                                        "per_database": failed}
            return self.calibration_report

        for p, r in zip(all_pairs, final_residuals, strict=True):
            p["residual"] = r
            p["cv_residual"] = cv_residual.get(id(p))
            p["cv_fold"] = cv_fold.get(id(p))

        self.element_offsets = final_offsets
        self.matched_pairs_ = all_pairs

        loco_report = {
            "n_chemsystems": len(chemsystems),
            "n_folds_skipped": n_folds_skipped,
            "n_test_skipped_total": n_test_skipped_total,
            "pooled_rmse_meV": loco_rmse_meV,
            "folds": fold_reports,
        }
        if loco_rmse_meV > 50:  # noqa: PLR2004
            loco_report["warning"] = f"Out-of-fold (LOCO) RMSE = {loco_rmse_meV:.0f} meV/atom exceeds 50 meV threshold."
        elif loco_rmse_meV > 25:  # noqa: PLR2004
            loco_report["warning"] = f"Out-of-fold (LOCO) RMSE = {loco_rmse_meV:.0f} meV/atom (above 25 meV DFT uncertainty)."

        self.calibration_report = {
            "status": "success",
            "reference": self.reference_database,
            "databases": non_ref_databases,
            "n_matches_total": len(all_pairs),
            "loco": loco_report,
            "final_fit": final_report,
            "per_database": {**self._per_database_breakdown(all_pairs, final_offsets), **failed},
        }

        logger.info(
            f"LOCO calibration {self.reference_database} ↔ {non_ref_databases} "  # noqa: G004
            f"over {len(chemsystems)} chemsystems: pooled out-of-fold RMSE = {loco_rmse_meV:.1f} meV/atom, "
            f"final-fit RMSE = {final_report['rmse_meV']:.1f} meV/atom ({len(all_pairs)} pairs total, "
            f"{n_folds_skipped} folds skipped, {n_test_skipped_total} pairs skipped for missing element coverage)"
        )

        self.is_trained = bool(self.element_offsets)
        return self.calibration_report

    # --- matching + fitting primitives ---

    def _find_matched_pairs(self, ref_db: str, other_db: str) -> list[dict]:  # noqa: C901
        matched_pairs = []
        seen_pairs = set()

        for chemsys, chemsys_dict in self.materials_dict.items():
            if ref_db not in chemsys_dict or other_db not in chemsys_dict:
                continue
            ref_materials = chemsys_dict[ref_db]
            other_materials = chemsys_dict[other_db]

            for ref_id, ref_mat in ref_materials.items():
                ref_attrs = ref_mat["normalized_attributes"]
                ref_ef = ref_attrs.get("formation_energy_per_atom")
                ref_formula = ref_attrs.get("reduced_formula")
                if ref_ef is None or ref_attrs.get("lattice_vectors") is None:
                    continue
                try:
                    ref_struct = convert_to_structure(ref_attrs)
                except (TypeError, KeyError, ValueError):
                    continue

                for other_id, other_mat in other_materials.items():
                    other_attrs = other_mat["normalized_attributes"]
                    other_ef = other_attrs.get("formation_energy_per_atom")
                    other_formula = other_attrs.get("reduced_formula")
                    if other_ef is None or other_attrs.get("lattice_vectors") is None:
                        continue
                    if ref_formula != other_formula:
                        continue
                    pair_key = (ref_id, other_id)
                    if pair_key in seen_pairs:
                        continue
                    try:
                        other_struct = convert_to_structure(other_attrs)
                    except (TypeError, KeyError, ValueError):
                        continue
                    try:
                        if self._matcher.fit(ref_struct, other_struct):
                            seen_pairs.add(pair_key)
                            comp = ref_attrs["composition"]
                            n_total = sum(comp.values())
                            fractions = {el: amt / n_total for el, amt in comp.items()}
                            matched_pairs.append({
                                "chemsys": chemsys, "ref_id": ref_id, "other_id": other_id,
                                "formula": ref_formula, "ref_ef": ref_ef, "other_ef": other_ef,
                                "delta": ref_ef - other_ef, "fractions": fractions,
                            })
                    except Exception:  # noqa: S112
                        continue
        return matched_pairs

    def _fit_offsets(self, matched_pairs: list[dict]
                     ) -> tuple[dict | None, list[float] | None, dict]:
        """Jointly fit per-(database, element) offsets via Huber regression.

        Does NOT mutate matched_pairs — LOCO refits the same pairs across
        many folds, so residuals are returned separately and callers decide
        how to attach them (final "residual" vs. a fold's "cv_residual").
        """
        columns = sorted({(p["database"], el) for p in matched_pairs for el in p["fractions"]})
        n_cols = len(columns)
        n_matches = len(matched_pairs)

        if n_matches < n_cols:
            return None, None, {
                "status": "failed",
                "reason": f"underdetermined: {n_matches} matches < {n_cols} (database, element) columns",
                "n_matches": n_matches, "n_elements": n_cols,
            }

        col_index = {col: j for j, col in enumerate(columns)}
        A = np.zeros((n_matches, n_cols))  # noqa: N806
        b = np.zeros(n_matches)
        for i, pair in enumerate(matched_pairs):
            b[i] = pair["delta"]
            db = pair["database"]
            for el, frac in pair["fractions"].items():
                A[i, col_index[(db, el)]] = frac

        model = HuberRegressor(epsilon=1.35, fit_intercept=False, alpha=0.01, max_iter=2000)
        model.fit(A, b)

        delta = model.coef_
        residuals_arr = b - A @ delta
        rmse = float(np.sqrt(np.mean(residuals_arr ** 2)))
        max_residual = float(np.max(np.abs(residuals_arr)))
        offsets = {col: float(delta[j]) for j, col in enumerate(columns)}
        residuals = [float(r) for r in residuals_arr]

        report = {
            "status": "success", "reference": self.reference_database,
            "n_matches": n_matches, "n_columns": n_cols,
            "rmse_meV": round(rmse * 1000, 1), "max_residual_meV": round(max_residual * 1000, 1),
        }
        if rmse > 0.050:  # noqa: PLR2004
            report["warning"] = f"Train RMSE = {rmse*1000:.0f} meV/atom exceeds 50 meV threshold."
        elif rmse > 0.025:  # noqa: PLR2004
            report["warning"] = f"Train RMSE = {rmse*1000:.0f} meV/atom (above 25 meV DFT uncertainty)."

        return offsets, residuals, report

    def _evaluate(self, matched_pairs: list[dict], offsets: dict) -> tuple[dict, list[float]]:
        """Evaluate fitted offsets on held-out pairs. Does not mutate matched_pairs."""
        if not matched_pairs:
            return {"status": "skipped", "reason": "no test pairs available", "n_matches": 0}, []

        residuals = []
        for pair in matched_pairs:
            db = pair["database"]
            predicted = sum(frac * offsets.get((db, el), 0.0) for el, frac in pair["fractions"].items())
            residuals.append(float(pair["delta"] - predicted))

        residuals_arr = np.array(residuals)
        rmse = float(np.sqrt(np.mean(residuals_arr ** 2)))
        report = {"status": "success", "n_matches": len(matched_pairs),
                  "rmse_meV": round(rmse * 1000, 1),
                  "max_residual_meV": round(float(np.max(np.abs(residuals_arr))) * 1000, 1)}
        return report, residuals

    def _per_database_breakdown(self, all_pairs: list[dict], offsets: dict) -> dict:
        databases = sorted({p["database"] for p in all_pairs})
        breakdown = {}
        for db in databases:
            db_pairs = [p for p in all_pairs if p["database"] == db]
            elements = sorted(el for (d, el) in offsets if d == db)
            cv_residuals = [p["cv_residual"] for p in db_pairs if p.get("cv_residual") is not None]
            final_residuals = [p["residual"] for p in db_pairs]
            breakdown[db] = {
                "elements": elements,
                "offsets_meV": {el: round(offsets[(db, el)] * 1000, 1) for el in elements},
                "n_matches": len(db_pairs),
                "n_cv_evaluated": len(cv_residuals),
                "cv_rmse_meV": self._rmse_meV(cv_residuals),
                "final_fit_rmse_meV": self._rmse_meV(final_residuals),
            }
        return breakdown

    @staticmethod
    def _rmse_meV(residuals: list[float]) -> float | None:  # noqa: N802
        if not residuals:
            return None
        residuals = np.array(residuals)
        return round(float(np.sqrt(np.mean(residuals ** 2))) * 1000, 1)

    def _apply_corrections(self, database: str, offsets: dict) -> None:
        for chemsys_dict in self.materials_dict.values():
            if database not in chemsys_dict:
                continue
            for material in chemsys_dict[database].values():
                attrs = material["normalized_attributes"]
                ef = attrs.get("formation_energy_per_atom")
                comp = attrs.get("composition")
                if ef is None or comp is None:
                    continue
                n_total = sum(comp.values())
                correction = sum((amt / n_total) * offsets.get(el, 0.0) for el, amt in comp.items())
                attrs["formation_energy_per_atom_uncorrected"] = ef
                attrs["formation_energy_per_atom"] = ef + correction
                attrs["calibration_correction"] = correction
                attrs["calibration_reference"] = self.reference_database

    def get_report_summary(self) -> str:
        report = self.calibration_report
        lines = [f"Cross-database calibration (reference: {self.reference_database})", "=" * 60]

        if report.get("status") != "success":
            lines.append(f"\nFAILED: {report.get('reason', 'unknown error')}")
            return "\n".join(lines)

        loco, final = report["loco"], report["final_fit"]
        lines.append(f"\nLeave-one-chemsys-out CV ({loco['n_chemsystems']} chemsystems, "
                      f"{len(report['databases'])} non-reference databases pooled):")
        lines.append(f"  Pooled out-of-fold RMSE: {loco['pooled_rmse_meV']} meV/atom")
        if loco["n_folds_skipped"]:
            lines.append(f"  WARNING: {loco['n_folds_skipped']}/{loco['n_chemsystems']} folds skipped (underdetermined)")
        if loco["n_test_skipped_total"]:
            lines.append(f"  {loco['n_test_skipped_total']} pairs skipped from evaluation (no element coverage)")
        if "warning" in loco:
            lines.append(f"  WARNING: {loco['warning']}")
        lines.append(f"  Final offsets fit on all {report['n_matches_total']} pairs: RMSE = {final['rmse_meV']} meV/atom")
        if "warning" in final:
            lines.append(f"  WARNING (final fit): {final['warning']}")

        for db, db_report in report["per_database"].items():
            lines.append(f"\n{db}:")
            if db_report.get("status") == "failed":
                lines.append(f"  FAILED: {db_report['reason']}")
                continue
            lines.append(f"  Elements fitted: {', '.join(db_report['elements'])}")
            lines.append(f"  Offsets:         {db_report['offsets_meV']} meV/atom")
            lines.append(f"  Matches:         {db_report['n_matches']} pairs "
                         f"({db_report['n_cv_evaluated']} evaluated out-of-fold)")
            lines.append(f"  CV RMSE:         {db_report['cv_rmse_meV']} meV/atom")
            lines.append(f"  Final-fit RMSE:  {db_report['final_fit_rmse_meV']} meV/atom")

        return "\n".join(lines)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np


def plot_calibration_parity(calibrator):
    """1x3 parity plot: uncorrected vs. final fit (in-sample) vs. CV fit (out-of-fold).

    Panels:
      1. "Before Calibration" — raw energies for every matched pair.
      2. "Final Fit" — all pairs, corrected using the final offsets
         (element_offsets, fit on all data). In-sample.
      3. "CV Fit (out-of-fold)" — all pairs, corrected using each pair's
         own out-of-fold prediction (cv_residual), for an honest read on
         how well the offsets generalize.
    """
    if not calibrator.matched_pairs_:
        raise ValueError("No matched pairs found. Did you call calibrator.train() first?")

    all_pairs = calibrator.matched_pairs_
    cv_pairs = [p for p in all_pairs if p.get("cv_residual") is not None]

    def final_corrected_other_ef(pair):
        db = pair["database"]
        correction = sum(frac * calibrator.element_offsets.get((db, el), 0.0)
                          for el, frac in pair["fractions"].items())
        return pair["other_ef"] + correction

    def cv_corrected_other_ef(pair):
        # ref_ef - cv_residual is guaranteed consistent with cv_residual,
        # since cv_residual = ref_ef - (other_ef + fold_correction).
        return pair["ref_ef"] - pair["cv_residual"]

    def rmse(residuals):
        residuals = np.array(residuals)
        return float(np.sqrt(np.mean(residuals ** 2))) if len(residuals) else float("nan")

    panels = [
        (
            "Before Calibration",
            [p["ref_ef"] for p in all_pairs],
            [p["other_ef"] for p in all_pairs],
            [abs(p["delta"]) for p in all_pairs],
            [p["delta"] for p in all_pairs],
            len(all_pairs),
        ),
        (
            "Final Fit",
            [p["ref_ef"] for p in all_pairs],
            [final_corrected_other_ef(p) for p in all_pairs],
            [abs(p["residual"]) for p in all_pairs],
            [p["residual"] for p in all_pairs],
            len(all_pairs),
        ),
        (
            "CV Fit (out-of-fold)",
            [p["ref_ef"] for p in cv_pairs],
            [cv_corrected_other_ef(p) for p in cv_pairs],
            [abs(p["cv_residual"]) for p in cv_pairs],
            [p["cv_residual"] for p in cv_pairs],
            len(cv_pairs),
        ),
    ]

    all_diffs = [d for _, _, _, diffs, _, _ in panels for d in diffs]
    vmin, vmax = 0, (max(all_diffs) if all_diffs else 1.0)

    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Charter", "Georgia", "DejaVu Serif"],
        "axes.labelsize": 11, "axes.titlesize": 12,
        "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "xtick.direction": "in", "ytick.direction": "in",
    })

    cmap = mpl.colormaps["plasma"]
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.5))

    for ax, (title, ref_energies, dup_energies, abs_diffs, signed_diffs, n) in zip(axes, panels, strict=True):
        if n == 0:
            ax.text(0.5, 0.5, "No pairs", ha="center", va="center",
                     transform=ax.transAxes, fontfamily="sans-serif")
            ax.set_title(title, fontfamily="sans-serif")
            continue

        ax.scatter(ref_energies, dup_energies, c=abs_diffs, cmap=cmap, norm=norm,
                   s=60, alpha=0.9, linewidths=0.8, edgecolors="black", zorder=3)
        ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))

        all_energies = ref_energies + dup_energies
        min_e, max_e = min(all_energies), max(all_energies)
        ax.plot([min_e, max_e], [min_e, max_e], color="#666666", linewidth=1.5, linestyle=(0, (4, 3)))

        rmse_val = rmse(signed_diffs)
        ax.text(0.05, 0.95, f"RMSE = {rmse_val:.4f} eV atom$^{{-1}}$\nn = {n}",
                transform=ax.transAxes, ha="left", va="top", fontsize=9, fontfamily="sans-serif",
                bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "0.8", "lw": 0.6})

        ax.set_xlabel(r"Reference $E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_ylabel(r"$E_\mathrm{f}$ (eV atom$^{-1}$)")
        ax.set_title(title, fontfamily="sans-serif")

    plt.tight_layout()
    cbar_ax = fig.add_axes([1.01, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
    cbar.set_label(r"$|\Delta E_\mathrm{f}|$ (eV atom$^{-1}$)", fontfamily="sans-serif", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    return fig

In [ ]:
import json
from pathlib import Path

extracted_materials = Path.cwd() / "data" / "extracted_materials.json"
with extracted_materials.open() as f:
    stability_data = json.load(f)

reference_database = "https://optimade.materialsproject.org/"
database_priority_list = [
    "https://optimade.materialsproject.org/",
    "https://alexandria.icams.rub.de/pbe",
    "https://oqmd.org/optimade/",
]

# Calibrate energy across databases (5-fold CV, Huber regression)
calibrator = EnergyCalibrator(stability_data, reference_database)
calibrator.train()
print(calibrator.get_report_summary())
energy_corrected_data = calibrator.fit()

# Sanity-check the fit: uncorrected vs. final fit (in-sample) vs. CV fit (out-of-fold)
fig = plot_calibration_parity(calibrator)
output_path = Path.cwd() / "figures" / "cross_database_comparison_cv_huber.pdf"
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)


import numpy as np
import networkx as nx
def tag_duplicate_structures(structures_by_chemsys: dict["str", list],
                             truth_matrices: dict["str", np.ndarray]) -> dict[str, list]:
    tagged_structures_by_chemsys = {}
    for chemsys, list_of_structures in structures_by_chemsys.items():
        truth_matrix = truth_matrices[chemsys]
        n = len(list_of_structures)
        # Build graph and find connected components
        G = nx.Graph()  # noqa: N806
        G.add_nodes_from(range(n))
        for i in range(n):
            for j in range(i + 1, n):
                if truth_matrix[i, j]:
                    G.add_edge(i, j)
        # Tag structures based on connected components
        for component in nx.connected_components(G):
            original_idx = min(component)
            original = list_of_structures[original_idx]
            for idx in component:
                structure = list_of_structures[idx]
                if len(component) == 1:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = None
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                elif idx == original_idx:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "original"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = None
                    structure["json_entry"]["normalized_attributes"]["original_database"] = None
                else:
                    structure["json_entry"]["normalized_attributes"]["duplicate_status"] = "duplicate"
                    structure["json_entry"]["normalized_attributes"]["original_material_id"] = original["json_entry"]["normalized_attributes"]["material_id"]
                    structure["json_entry"]["normalized_attributes"]["original_database"] = original["json_entry"]["normalized_attributes"]["database"]
        tagged_structures_by_chemsys[chemsys] = list_of_structures
    return tagged_structures_by_chemsys


# Tag duplicates and originals
from matcollect.core.duplicate_removal.duplicate_remover import DuplicateRemover
duplicate_remover = DuplicateRemover(energy_corrected_data, database_priority_list)
structures_by_chemsys = duplicate_remover._generate_pymatgen_structures_by_chemsys()
duplicate_remover.truth_matrices = duplicate_remover._get_structure_similarities(structures_by_chemsys,
                                                       duplicate_remover.tolerances)
tagged_structures_by_chemsys = tag_duplicate_structures(structures_by_chemsys,
                                                        duplicate_remover.truth_matrices)
tagged_materials = duplicate_remover._parse_structures(tagged_structures_by_chemsys)